In [1]:
!pip install pandas numpy scikit-learn matplotlib seaborn nltk spacy tensorflow transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 kB 1.9 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of mkl-fft to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of mkl-fft to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 47.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 101.8 MB/s eta 0:00:0000:010:01
  Attempting uninstall: blis
    Found existing installation: blis 1.3.0
    Uninstalling blis-1.3.0:
      Successfully uninstalled blis-1.3.0
  Attempting uninstall: thinc
    Found existing installation: t

In [5]:
# ------------------------------------------------------------
# 0) Imports
# ------------------------------------------------------------
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from sklearn.metrics import classification_report
import pandas as pd

# ------------------------------------------------------------
# 1) Load data
# ------------------------------------------------------------
# Change this to your dataset path
DATA_PATH = "/kaggle/input/learning/IMDB_Cleaned (1).csv"  

df = pd.read_csv(DATA_PATH)

# Fallback to 'review' if 'cleaned_review' doesn't exist
text_column = 'cleaned_review' if 'cleaned_review' in df.columns else 'review'

# Parameters
MAX_VOCAB = 20000
MAX_LEN = 200

# ------------------------------------------------------------
# 2) Tokenization
# ------------------------------------------------------------
tokenizer = Tokenizer(num_words=MAX_VOCAB, oov_token="<OOV>")
tokenizer.fit_on_texts(df[text_column])

sequences = tokenizer.texts_to_sequences(df[text_column])
X = pad_sequences(sequences, maxlen=MAX_LEN, padding='post', truncating='post')

# Encode labels
le = LabelEncoder()
y = le.fit_transform(df['sentiment'])  # 0 = negative, 1 = positive

# ------------------------------------------------------------
# 3) Split: 40k train, 5k val, 5k test
# ------------------------------------------------------------
# First split: Train (40k) + Temp (10k)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, train_size=40000, random_state=42, stratify=y
)

# Second split: Validation (5k) + Test (5k)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print("Train size:", X_train.shape)
print("Validation size:", X_val.shape)
print("Test size:", X_test.shape)

# ------------------------------------------------------------
# 4) Build Model
# ------------------------------------------------------------
model = Sequential([
    Embedding(input_dim=MAX_VOCAB, output_dim=128, input_length=MAX_LEN),
    LSTM(64, return_sequences=False),
    Dropout(0.5),
    Dense(32, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

early_stop = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)

# Explicitly build
model.build(input_shape=(None, MAX_LEN))
model.summary()

# ------------------------------------------------------------
# 5) Train Model
# ------------------------------------------------------------
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=20,
    batch_size=128,
    callbacks=[early_stop]
)

# ------------------------------------------------------------
# 6) Validation Classification Report
# ------------------------------------------------------------
y_val_pred_probs = model.predict(X_val)
y_val_pred = (y_val_pred_probs > 0.5).astype(int)

print("\n📊 Validation Classification Report:\n")
print(classification_report(y_val, y_val_pred, target_names=le.classes_))

# ------------------------------------------------------------
# 7) Test Classification Report
# ------------------------------------------------------------
y_test_pred_probs = model.predict(X_test)
y_test_pred = (y_test_pred_probs > 0.5).astype(int)

print("\n📊 Test Classification Report:\n")
print(classification_report(y_test, y_test_pred, target_names=le.classes_))


Train size: (40000, 200)
Validation size: (5000, 200)
Test size: (5000, 200)


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ (None, 200, 128)       │     2,560,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,611,521 (9.96 MB)

 Trainable params: 2,611,521 (9.96 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 8s 15ms/step - accuracy: 0.5079 - loss: 0.6931 - val_accuracy: 0.5394 - val_loss: 0.6862
Epoch 2/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 13ms/step - accuracy: 0.5913 - loss: 0.6514 - val_accuracy: 0.6162 - val_loss: 0.6075
Epoch 3/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 13ms/step - accuracy: 0.6512 - loss: 0.5546 - val_accuracy: 0.8360 - val_loss: 0.5405
Epoch 4/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 13ms/step - accuracy: 0.8609 - loss: 0.3660 - val_accuracy: 0.8672 - val_loss: 0.3324
Epoch 5/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 13ms/step - accuracy: 0.9184 - loss: 0.2313 - val_accuracy: 0.8710 - val_loss: 0.3448
Epoch 6/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 13ms/step - accuracy: 0.9485 - loss: 0.1629 - val_accuracy: 0.8780 - val_loss: 0.3695
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step

📊 Validation Classification Report:

              precision    recall  f1-score   support

    negative       0.88      0.85      0.86      2500
    positive       0.85      0.89     

In [6]:
# ------------------------------------------------------------
# 0) Imports
# ------------------------------------------------------------
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional, Input, Layer
from sklearn.metrics import classification_report
import tensorflow as tf
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1) Load data
# ------------------------------------------------------------
DATA_PATH = "/kaggle/input/learning/IMDB_Cleaned (1).csv"  # <-- change to your file path
df = pd.read_csv(DATA_PATH)

# Fallback to 'review' if 'cleaned_review' doesn't exist
text_column = 'cleaned_review' if 'cleaned_review' in df.columns else 'review'

# Parameters
MAX_VOCAB = 20000
MAX_LEN = 200

# ------------------------------------------------------------
# 2) Tokenization
# ------------------------------------------------------------
tokenizer = Tokenizer(num_words=MAX_VOCAB, oov_token="<OOV>")
tokenizer.fit_on_texts(df[text_column])

sequences = tokenizer.texts_to_sequences(df[text_column])
X = pad_sequences(sequences, maxlen=MAX_LEN, padding='post', truncating='post')

# Encode labels
le = LabelEncoder()
y = le.fit_transform(df['sentiment'])  # 0 = negative, 1 = positive

# ------------------------------------------------------------
# 3) Split: 40k train, 5k val, 5k test
# ------------------------------------------------------------
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, train_size=40000, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print("Train size:", X_train.shape)
print("Validation size:", X_val.shape)
print("Test size:", X_test.shape)

# ------------------------------------------------------------
# 4) Attention Layer
# ------------------------------------------------------------
class Attention(Layer):
    def __init__(self, **kwargs):
        super(Attention, self).__init__(**kwargs)

    def build(self, input_shape):
        self.W = self.add_weight(name="att_weight", shape=(input_shape[-1], 1),
                                 initializer="normal")
        self.b = self.add_weight(name="att_bias", shape=(input_shape[1], 1),
                                 initializer="zeros")        
        super(Attention, self).build(input_shape)

    def call(self, x):
        e = tf.keras.backend.tanh(tf.keras.backend.dot(x, self.W) + self.b)
        a = tf.keras.backend.softmax(e, axis=1)
        output = x * a
        return tf.keras.backend.sum(output, axis=1)

# ------------------------------------------------------------
# 5) Build BiLSTM + Attention Model
# ------------------------------------------------------------
inputs = Input(shape=(MAX_LEN,))
embedding = Embedding(input_dim=MAX_VOCAB, output_dim=128, input_length=MAX_LEN)(inputs)
bilstm = Bidirectional(LSTM(64, return_sequences=True))(embedding)
attention = Attention()(bilstm)
drop1 = Dropout(0.5)(attention)
dense1 = Dense(32, activation="relu")(drop1)
drop2 = Dropout(0.3)(dense1)
outputs = Dense(1, activation="sigmoid")(drop2)

model = Model(inputs=inputs, outputs=outputs)

model.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])
early_stop = EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True)

model.summary()

# ------------------------------------------------------------
# 6) Train Model
# ------------------------------------------------------------
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=128,
    callbacks=[early_stop]
)

# ------------------------------------------------------------
# 7) Validation Classification Report
# ------------------------------------------------------------
y_val_pred_probs = model.predict(X_val)
y_val_pred = (y_val_pred_probs > 0.5).astype(int)

print("\n📊 Validation Classification Report:\n")
print(classification_report(y_val, y_val_pred, target_names=le.classes_))

# ------------------------------------------------------------
# 8) Test Classification Report
# ------------------------------------------------------------
y_test_pred_probs = model.predict(X_test)
y_test_pred = (y_test_pred_probs > 0.5).astype(int)

print("\n📊 Test Classification Report:\n")
print(classification_report(y_test, y_test_pred, target_names=le.classes_))


Train size: (40000, 200)
Validation size: (5000, 200)
Test size: (5000, 200)


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_4 (InputLayer)      │ (None, 200)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_4 (Embedding)         │ (None, 200, 128)       │     2,560,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 200, 128)       │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ attention (Attention)           │ (None, 128)            │           328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 32)             │         4,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_9 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,663,305 (10.16 MB)

 Trainable params: 2,663,305 (10.16 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 13s 27ms/step - accuracy: 0.6879 - loss: 0.5465 - val_accuracy: 0.8768 - val_loss: 0.2903
Epoch 2/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 8s 26ms/step - accuracy: 0.9164 - loss: 0.2267 - val_accuracy: 0.8818 - val_loss: 0.2796
Epoch 3/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 8s 26ms/step - accuracy: 0.9449 - loss: 0.1578 - val_accuracy: 0.8818 - val_loss: 0.3036
Epoch 4/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 8s 26ms/step - accuracy: 0.9669 - loss: 0.0997 - val_accuracy: 0.8670 - val_loss: 0.4396
157/157 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step

📊 Validation Classification Report:

              precision    recall  f1-score   support

    negative       0.92      0.84      0.88      2500
    positive       0.85      0.92      0.89      2500

    accuracy                           0.88      5000
   macro avg       0.88      0.88      0.88      5000
weighted avg       0.88      0.88      0.88      5000

157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step

📊 Test Classification Report:

    

In [11]:
from tensorflow.keras.layers import GRU, Embedding, Dropout, Dense
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import classification_report

gru_model = Sequential([
    Embedding(input_dim=MAX_VOCAB, output_dim=128, input_shape=(MAX_LEN,)),  # ✅ fixed
    GRU(64, return_sequences=False),
    Dropout(0.5),
    Dense(32, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

gru_model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

early_stop = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)

gru_model.summary()  # ✅ now will show shapes + parameters

# Train GRU
history_gru = gru_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=128,
    callbacks=[early_stop]
)

# Validation classification report
y_val_pred_probs = gru_model.predict(X_val)
y_val_pred = (y_val_pred_probs > 0.5).astype(int)
print("\n📊 GRU Validation Report:\n", classification_report(y_val, y_val_pred, target_names=le.classes_))

# Test classification report
y_test_pred_probs = gru_model.predict(X_test)
y_test_pred = (y_test_pred_probs > 0.5).astype(int)
print("\n📊 GRU Test Report:\n", classification_report(y_test, y_test_pred, target_names=le.classes_))


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_9 (Embedding)         │ (None, 200, 128)       │     2,560,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_3 (GRU)                     │ (None, 64)             │        37,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_18 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_18 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_19 (Dropout)            │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_19 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,599,361 (9.92 MB)

 Trainable params: 2,599,361 (9.92 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 7s 14ms/step - accuracy: 0.4975 - loss: 0.6935 - val_accuracy: 0.5744 - val_loss: 0.6813
Epoch 2/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 13ms/step - accuracy: 0.6053 - loss: 0.6518 - val_accuracy: 0.8186 - val_loss: 0.4166
Epoch 3/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 13ms/step - accuracy: 0.8677 - loss: 0.3469 - val_accuracy: 0.8890 - val_loss: 0.2749
Epoch 4/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 13ms/step - accuracy: 0.9292 - loss: 0.2059 - val_accuracy: 0.8866 - val_loss: 0.2798
Epoch 5/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 13ms/step - accuracy: 0.9619 - loss: 0.1235 - val_accuracy: 0.8852 - val_loss: 0.3408
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step

📊 GRU Validation Report:
               precision    recall  f1-score   support

    negative       0.90      0.88      0.89      2500
    positive       0.88      0.90      0.89      2500

    accuracy                           0.89      5000
   macro avg       0.89      0.89      0.89      5000
weighted avg   

In [13]:
# ------------------------------------------------------------
# CNN Model
# ------------------------------------------------------------
from tensorflow.keras.layers import Conv1D, GlobalMaxPooling1D, Embedding, Dropout, Dense
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import classification_report

cnn_model = Sequential([
    Embedding(input_dim=MAX_VOCAB, output_dim=128, input_shape=(MAX_LEN,)),  # ✅ fixed
    Conv1D(filters=128, kernel_size=5, activation='relu'),
    GlobalMaxPooling1D(),
    Dropout(0.5),
    Dense(32, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

cnn_model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

early_stop = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)

cnn_model.summary()  # ✅ will now show shapes and params

# Train CNN
history_cnn = cnn_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=128,
    callbacks=[early_stop]
)

# Validation classification report
y_val_pred_probs = cnn_model.predict(X_val)
y_val_pred = (y_val_pred_probs > 0.5).astype(int)
print("\n📊 CNN Validation Report:\n", classification_report(y_val, y_val_pred, target_names=le.classes_))

# Test classification report
y_test_pred_probs = cnn_model.predict(X_test)
y_test_pred = (y_test_pred_probs > 0.5).astype(int)
print("\n📊 CNN Test Report:\n", classification_report(y_test, y_test_pred, target_names=le.classes_))


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_10"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_11 (Embedding)        │ (None, 200, 128)       │     2,560,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ (None, 196, 128)       │        82,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d_2          │ (None, 128)            │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_22 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_22 (Dense)                │ (None, 32)             │         4,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_23 (Dropout)            │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_23 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,646,209 (10.09 MB)

 Trainable params: 2,646,209 (10.09 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 7s 13ms/step - accuracy: 0.6256 - loss: 0.6200 - val_accuracy: 0.8510 - val_loss: 0.3383
Epoch 2/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.8759 - loss: 0.3142 - val_accuracy: 0.8664 - val_loss: 0.3071
Epoch 3/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.9342 - loss: 0.1853 - val_accuracy: 0.8658 - val_loss: 0.3353
Epoch 4/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.9667 - loss: 0.1053 - val_accuracy: 0.8680 - val_loss: 0.3887
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step

📊 CNN Validation Report:
               precision    recall  f1-score   support

    negative       0.86      0.87      0.87      2500
    positive       0.87      0.86      0.87      2500

    accuracy                           0.87      5000
   macro avg       0.87      0.87      0.87      5000
weighted avg       0.87      0.87      0.87      5000

157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step

📊 CNN Test Report:
               precision    rec